In [ ]:
# Celda 1: Leer Excel y generar CSV resumen de hojas/columnas

import pandas as pd
from pathlib import Path
import os

# Ruta de la carpeta donde está el Excel
carpeta_data = Path(
    r"C:\Users\isancheza\OneDrive - SBS\Documentos\UPC\Data Vizualitation\7304416-base-de-datos-sidpol-a-setiembre-2025(2) (1)\Data"
)

# Buscar automáticamente el archivo Excel de SIDPOL
archivos_excel = list(carpeta_data.glob("*.xlsx"))

archivo_excel = None
for archivo in archivos_excel:
    if "SIDPOL" in archivo.name.upper():
        archivo_excel = archivo
        break

if archivo_excel is None:
    raise FileNotFoundError("No se encontró un archivo Excel que contenga 'SIDPOL' en el nombre.")

# Crear carpeta de salida
carpeta_salida = carpeta_data / "salida_revision"
carpeta_salida.mkdir(exist_ok=True)

# Leer archivo Excel
excel = pd.ExcelFile(archivo_excel)

# Diccionario para guardar cada hoja leída
diccionario_hojas = {}

# Lista para construir el resumen
resumen_columnas = []

for hoja in excel.sheet_names:
    df_temp = pd.read_excel(archivo_excel, sheet_name=hoja)
    diccionario_hojas[hoja] = df_temp
    
    total_filas = df_temp.shape[0]
    total_columnas = df_temp.shape[1]
    
    for orden_columna, columna in enumerate(df_temp.columns, start=1):
        serie = df_temp[columna]
        
        valores_ejemplo = (
            serie
            .dropna()
            .astype(str)
            .drop_duplicates()
            .head(5)
            .tolist()
        )
        
        resumen_columnas.append({
            "hoja": hoja,
            "filas_hoja": total_filas,
            "columnas_hoja": total_columnas,
            "orden_columna": orden_columna,
            "nombre_columna": columna,
            "tipo_dato": str(serie.dtype),
            "valores_nulos": int(serie.isna().sum()),
            "porcentaje_nulos": round(serie.isna().mean() * 100, 2),
            "valores_unicos": int(serie.nunique(dropna=True)),
            "ejemplos_valores": " | ".join(valores_ejemplo)
        })

# Convertir resumen a DataFrame
df_resumen_columnas = pd.DataFrame(resumen_columnas)

# Guardar CSV resumen
ruta_resumen_csv = carpeta_salida / "resumen_columnas_por_hoja.csv"

df_resumen_columnas.to_csv(
    ruta_resumen_csv,
    index=False,
    encoding="utf-8-sig"
)

# Abrir automáticamente el CSV en Windows
os.startfile(ruta_resumen_csv)

In [ ]:
# Celda 2: Limpieza general de todas las hojas SIDPOL

import pandas as pd
import numpy as np
from pathlib import Path
import re
import unicodedata
import os
from functools import reduce

# =========================
# 1. Rutas
# =========================

carpeta_data = Path(
    r"C:\Users\isancheza\OneDrive - SBS\Documentos\UPC\Data Vizualitation\7304416-base-de-datos-sidpol-a-setiembre-2025(2) (1)\Data"
)

archivos_excel = list(carpeta_data.glob("*.xlsx"))

archivo_excel = None
for archivo in archivos_excel:
    if "SIDPOL" in archivo.name.upper():
        archivo_excel = archivo
        break

if archivo_excel is None:
    raise FileNotFoundError("No se encontró el archivo Excel de SIDPOL.")

carpeta_salida = carpeta_data / "salida_limpieza_sidpol"
carpeta_salida.mkdir(exist_ok=True)

carpeta_hojas_limpias = carpeta_salida / "hojas_limpias"
carpeta_hojas_limpias.mkdir(exist_ok=True)

# =========================
# 2. Funciones de limpieza
# =========================

def quitar_tildes(texto):
    if pd.isna(texto):
        return np.nan
    
    texto = str(texto)
    texto = unicodedata.normalize("NFKD", texto)
    texto = "".join([c for c in texto if not unicodedata.combining(c)])
    return texto


def limpiar_texto(texto):
    """
    Normaliza textos:
    - quita tildes
    - convierte a mayúsculas
    - elimina espacios dobles
    - limpia espacios al inicio y final
    """
    if pd.isna(texto):
        return np.nan
    
    texto = quitar_tildes(texto)
    texto = texto.upper()
    texto = texto.strip()
    texto = re.sub(r"\s+", " ", texto)
    return texto


def limpiar_nombre_columna(columna):
    """
    Normaliza nombres de columnas.
    """
    columna = quitar_tildes(columna)
    columna = columna.upper().strip()
    columna = re.sub(r"\s+", "_", columna)
    columna = columna.replace(".", "_")
    columna = columna.replace("-", "_")
    columna = re.sub(r"_+", "_", columna)
    return columna


def normalizar_ubigeo(valor):
    """
    Convierte UBIGEO a texto de 6 dígitos.
    Ejemplo:
    10101 -> 010101
    """
    if pd.isna(valor):
        return np.nan
    
    try:
        valor = str(int(float(valor)))
    except:
        valor = str(valor).strip()
    
    valor = re.sub(r"\D", "", valor)
    
    if valor == "":
        return np.nan
    
    return valor.zfill(6)


def crear_fecha(anio, mes):
    """
    Crea fecha usando año y mes.
    Día fijo = 1.
    """
    return pd.to_datetime(
        dict(year=anio, month=mes, day=1),
        errors="coerce"
    )


def limpiar_hoja_sidpol(df, nombre_hoja):
    """
    Limpia una hoja del Excel SIDPOL.
    """
    
    df = df.copy()
    
    # Normalizar nombres de columnas
    df.columns = [limpiar_nombre_columna(c) for c in df.columns]
    
    # Renombrar columnas a nombres estándar
    mapa_columnas = {
        "ANIO": "anio",
        "MES": "mes",
        "DPTO_HECHO_NEW": "departamento",
        "PROV_HECHO": "provincia",
        "DIST_HECHO": "distrito",
        "UBIGEO_HECHO": "ubigeo",
        "N_DIST_ID_DGC": "conteo",
        "ES_DELITO_X": "es_delito",
        "PRINCIPALES_TIPOS": "principales_tipos",
        "PMODALIDADES": "p_modalidad",
        "P_MODALIDADES": "p_modalidad",
        "TIPO": "tipo",
        "SUB_TIPO": "sub_tipo",
        "MODALIDAD": "modalidad",
        "DIST_EMERGENCIA": "dist_emergencia"
    }
    
    df = df.rename(columns=mapa_columnas)
    
    # Agregar hoja de origen
    df["fuente_hoja"] = nombre_hoja
    
    # Convertir año y mes
    if "anio" in df.columns:
        df["anio"] = pd.to_numeric(df["anio"], errors="coerce").astype("Int64")
    
    if "mes" in df.columns:
        df["mes"] = pd.to_numeric(df["mes"], errors="coerce").astype("Int64")
    
    # Convertir conteo
    if "conteo" in df.columns:
        df["conteo"] = pd.to_numeric(df["conteo"], errors="coerce").fillna(0).astype(int)
    
    # Normalizar UBIGEO
    if "ubigeo" in df.columns:
        df["ubigeo"] = df["ubigeo"].apply(normalizar_ubigeo)
    
    # Normalizar textos
    columnas_texto = [
        "departamento",
        "provincia",
        "distrito",
        "es_delito",
        "principales_tipos",
        "p_modalidad",
        "tipo",
        "sub_tipo",
        "modalidad"
    ]
    
    for col in columnas_texto:
        if col in df.columns:
            df[col] = df[col].apply(limpiar_texto)
    
    # Crear fecha
    if "anio" in df.columns and "mes" in df.columns:
        df["fecha"] = crear_fecha(df["anio"], df["mes"])
        df["trimestre"] = df["fecha"].dt.quarter.astype("Int64")
        df["anio_mes"] = df["fecha"].dt.strftime("%Y-%m")
    
    # Crear nivel geográfico
    if "ubigeo" in df.columns:
        df["nivel_geografico"] = "DISTRITO"
    else:
        df["nivel_geografico"] = "DEPARTAMENTO"
    
    # Crear columnas vacías si no existen para estandarizar estructura
    columnas_base = [
        "fuente_hoja",
        "nivel_geografico",
        "anio",
        "mes",
        "fecha",
        "anio_mes",
        "trimestre",
        "ubigeo",
        "departamento",
        "provincia",
        "distrito",
        "dist_emergencia",
        "es_delito",
        "principales_tipos",
        "p_modalidad",
        "tipo",
        "sub_tipo",
        "modalidad",
        "conteo"
    ]
    
    for col in columnas_base:
        if col not in df.columns:
            df[col] = np.nan
    
    # Ordenar columnas
    df = df[columnas_base]
    
    # Eliminar filas sin año, mes o conteo
    df = df.dropna(subset=["anio", "mes"])
    df = df[df["conteo"] >= 0]
    
    return df


def eliminar_duplicados_sin_sumar(df):
    """
    Elimina duplicados conservando el mayor conteo.
    No suma conteos.
    """
    
    columnas_llave = [c for c in df.columns if c != "conteo"]
    
    df_limpio = (
        df
        .groupby(columnas_llave, dropna=False, as_index=False)["conteo"]
        .max()
    )
    
    return df_limpio


# =========================
# 3. Leer, limpiar y guardar cada hoja
# =========================

excel = pd.ExcelFile(archivo_excel)

hojas_limpias = {}
qa_limpieza = []

for hoja in excel.sheet_names:
    df_original = pd.read_excel(archivo_excel, sheet_name=hoja)
    
    filas_originales = len(df_original)
    
    df_limpio = limpiar_hoja_sidpol(df_original, hoja)
    filas_despues_limpieza = len(df_limpio)
    
    df_limpio_sin_duplicados = eliminar_duplicados_sin_sumar(df_limpio)
    filas_finales = len(df_limpio_sin_duplicados)
    
    duplicados_eliminados = filas_despues_limpieza - filas_finales
    
    hojas_limpias[hoja] = df_limpio_sin_duplicados
    
    # Guardar cada hoja limpia en CSV
    nombre_archivo_csv = f"{hoja.replace('.', '_')}_limpia.csv"
    ruta_csv = carpeta_hojas_limpias / nombre_archivo_csv
    
    df_limpio_sin_duplicados.to_csv(
        ruta_csv,
        index=False,
        encoding="utf-8-sig"
    )
    
    qa_limpieza.append({
        "hoja": hoja,
        "filas_originales": filas_originales,
        "filas_despues_limpieza": filas_despues_limpieza,
        "filas_finales_sin_duplicados": filas_finales,
        "duplicados_eliminados": duplicados_eliminados,
        "columnas_finales": len(df_limpio_sin_duplicados.columns),
        "nivel_geografico": df_limpio_sin_duplicados["nivel_geografico"].dropna().unique()[0],
        "anio_min": df_limpio_sin_duplicados["anio"].min(),
        "anio_max": df_limpio_sin_duplicados["anio"].max(),
        "conteo_total_referencial": df_limpio_sin_duplicados["conteo"].sum()
    })

df_qa_limpieza = pd.DataFrame(qa_limpieza)

ruta_qa = carpeta_salida / "qa_limpieza_por_hoja.csv"

df_qa_limpieza.to_csv(
    ruta_qa,
    index=False,
    encoding="utf-8-sig"
)

os.startfile(carpeta_salida)

In [ ]:
# Celda 3: Crear tabla larga unificada

def construir_tabla_larga(hojas_limpias):
    tablas = []
    
    for hoja, df in hojas_limpias.items():
        df = df.copy()
        
        # Temp2: clasificación general ES_DELITO
        if hoja == "Temp2":
            temp = df.copy()
            temp["nivel_categoria"] = "ES_DELITO"
            temp["categoria_analitica"] = temp["es_delito"]
            tablas.append(temp)
        
        # Temp3 y Temp5: principales tipos
        elif hoja in ["Temp3", "Temp5"]:
            temp = df.copy()
            temp["nivel_categoria"] = "PRINCIPALES_TIPOS"
            temp["categoria_analitica"] = temp["principales_tipos"]
            tablas.append(temp)
        
        # Temp4 y Temp5.2: principales modalidades
        elif hoja in ["Temp4", "Temp5.2"]:
            temp = df.copy()
            temp["nivel_categoria"] = "P_MODALIDAD"
            temp["categoria_analitica"] = temp["p_modalidad"]
            tablas.append(temp)
        
        # Temp6 y Temp7: detalle completo tipo, subtipo y modalidad
        elif hoja in ["Temp6", "Temp7"]:
            temp = df.copy()
            temp["nivel_categoria"] = "DETALLE_TIPO_SUBTIPO_MODALIDAD"
            temp["categoria_analitica"] = temp["modalidad"]
            tablas.append(temp)
    
    df_largo = pd.concat(tablas, ignore_index=True)
    
    columnas_finales = [
        "fuente_hoja",
        "nivel_geografico",
        "nivel_categoria",
        "anio",
        "mes",
        "fecha",
        "anio_mes",
        "trimestre",
        "ubigeo",
        "departamento",
        "provincia",
        "distrito",
        "dist_emergencia",
        "es_delito",
        "principales_tipos",
        "p_modalidad",
        "tipo",
        "sub_tipo",
        "modalidad",
        "categoria_analitica",
        "conteo"
    ]
    
    df_largo = df_largo[columnas_finales]
    
    # Eliminar duplicados exactos en la tabla larga conservando mayor conteo
    columnas_llave = [c for c in df_largo.columns if c != "conteo"]
    
    df_largo = (
        df_largo
        .groupby(columnas_llave, dropna=False, as_index=False)["conteo"]
        .max()
    )
    
    return df_largo


df_sidpol_largo = construir_tabla_larga(hojas_limpias)

ruta_largo = carpeta_salida / "sidpol_largo_unificado_limpio.csv"

df_sidpol_largo.to_csv(
    ruta_largo,
    index=False,
    encoding="utf-8-sig"
)

In [ ]:
# Celda 4: Crear tabla ancha tipo super merge sin multiplicar filas

def nombre_seguro_columna(texto):
    """
    Convierte una categoría en nombre de columna seguro.
    """
    if pd.isna(texto):
        return "SIN_CATEGORIA"
    
    texto = limpiar_texto(texto)
    texto = re.sub(r"[^A-Z0-9]+", "_", texto)
    texto = re.sub(r"_+", "_", texto)
    texto = texto.strip("_")
    
    if texto == "":
        texto = "SIN_CATEGORIA"
    
    return texto


def crear_pivot_seguro(df, columna_categoria, prefijo):
    """
    Crea una tabla ancha por distrito-mes.
    Usa max para evitar sumar duplicados.
    """
    
    df = df.copy()
    df = df[df["nivel_geografico"] == "DISTRITO"]
    
    df = df.dropna(subset=["ubigeo", columna_categoria])
    
    if df.empty:
        return pd.DataFrame()
    
    llaves = [
        "anio",
        "mes",
        "fecha",
        "anio_mes",
        "trimestre",
        "ubigeo",
        "departamento",
        "provincia",
        "distrito"
    ]
    
    if "dist_emergencia" in df.columns:
        # Se agrega después como max por llave
        pass
    
    df["categoria_columna"] = (
        prefijo
        + "__"
        + df[columna_categoria].apply(nombre_seguro_columna)
    )
    
    pivot = (
        df
        .pivot_table(
            index=llaves,
            columns="categoria_columna",
            values="conteo",
            aggfunc="max",
            fill_value=0
        )
        .reset_index()
    )
    
    pivot.columns.name = None
    
    return pivot


# =========================
# 1. Base geográfica distrito-mes
# =========================

df_distrito = pd.concat(
    [
        hojas_limpias[h]
        for h in hojas_limpias.keys()
        if "ubigeo" in hojas_limpias[h].columns
    ],
    ignore_index=True
)

df_distrito = df_distrito[df_distrito["nivel_geografico"] == "DISTRITO"]

llaves_base = [
    "anio",
    "mes",
    "fecha",
    "anio_mes",
    "trimestre",
    "ubigeo",
    "departamento",
    "provincia",
    "distrito"
]

df_base_distrito_mes = (
    df_distrito[llaves_base + ["dist_emergencia"]]
    .drop_duplicates()
    .groupby(llaves_base, dropna=False, as_index=False)["dist_emergencia"]
    .max()
)

# =========================
# 2. Pivots por tipo de información
# =========================

tablas_pivot = []

# Principales tipos: Temp5
if "Temp5" in hojas_limpias:
    pivot_pt = crear_pivot_seguro(
        hojas_limpias["Temp5"],
        columna_categoria="principales_tipos",
        prefijo="PT"
    )
    if not pivot_pt.empty:
        tablas_pivot.append(pivot_pt)

# Principales modalidades: Temp5.2
if "Temp5.2" in hojas_limpias:
    pivot_pm = crear_pivot_seguro(
        hojas_limpias["Temp5.2"],
        columna_categoria="p_modalidad",
        prefijo="PM"
    )
    if not pivot_pm.empty:
        tablas_pivot.append(pivot_pm)

# Tipo detallado: Temp6 y Temp7
df_detalle = pd.concat(
    [
        hojas_limpias[h]
        for h in ["Temp6", "Temp7"]
        if h in hojas_limpias
    ],
    ignore_index=True
)

if not df_detalle.empty:
    pivot_tipo = crear_pivot_seguro(
        df_detalle,
        columna_categoria="tipo",
        prefijo="TIPO"
    )
    if not pivot_tipo.empty:
        tablas_pivot.append(pivot_tipo)

    pivot_subtipo = crear_pivot_seguro(
        df_detalle,
        columna_categoria="sub_tipo",
        prefijo="SUBTIPO"
    )
    if not pivot_subtipo.empty:
        tablas_pivot.append(pivot_subtipo)

    pivot_modalidad = crear_pivot_seguro(
        df_detalle,
        columna_categoria="modalidad",
        prefijo="MOD"
    )
    if not pivot_modalidad.empty:
        tablas_pivot.append(pivot_modalidad)

# =========================
# 3. Super merge seguro
# =========================

df_sidpol_super_merge = df_base_distrito_mes.copy()

for tabla in tablas_pivot:
    df_sidpol_super_merge = df_sidpol_super_merge.merge(
        tabla,
        on=llaves_base,
        how="outer"
    )

# Rellenar nulos numéricos con 0
columnas_no_numericas = llaves_base + ["dist_emergencia"]

columnas_metricas = [
    c for c in df_sidpol_super_merge.columns
    if c not in columnas_no_numericas
]

df_sidpol_super_merge[columnas_metricas] = (
    df_sidpol_super_merge[columnas_metricas]
    .fillna(0)
    .astype(int)
)

# Si dist_emergencia queda nulo, dejarlo como 0
df_sidpol_super_merge["dist_emergencia"] = (
    df_sidpol_super_merge["dist_emergencia"]
    .fillna(0)
    .astype(int)
)

# Ordenar
df_sidpol_super_merge = df_sidpol_super_merge.sort_values(
    ["anio", "mes", "departamento", "provincia", "distrito"]
).reset_index(drop=True)

ruta_super_merge = carpeta_salida / "sidpol_super_merge_distrito_mes.csv"

df_sidpol_super_merge.to_csv(
    ruta_super_merge,
    index=False,
    encoding="utf-8-sig"
)

In [ ]:
# Celda 5: QA del resultado final

qa_final = {}

qa_final["filas_sidpol_largo"] = len(df_sidpol_largo)
qa_final["columnas_sidpol_largo"] = len(df_sidpol_largo.columns)

qa_final["filas_super_merge"] = len(df_sidpol_super_merge)
qa_final["columnas_super_merge"] = len(df_sidpol_super_merge.columns)

llave_super_merge = [
    "anio",
    "mes",
    "ubigeo"
]

duplicados_llave = df_sidpol_super_merge.duplicated(subset=llave_super_merge).sum()

qa_final["duplicados_en_super_merge_por_anio_mes_ubigeo"] = int(duplicados_llave)

qa_final["anio_min_super_merge"] = int(df_sidpol_super_merge["anio"].min())
qa_final["anio_max_super_merge"] = int(df_sidpol_super_merge["anio"].max())

qa_final["distritos_unicos_super_merge"] = int(df_sidpol_super_merge["ubigeo"].nunique())
qa_final["departamentos_unicos_super_merge"] = int(df_sidpol_super_merge["departamento"].nunique())

qa_final["columnas_metricas_creadas"] = len([
    c for c in df_sidpol_super_merge.columns
    if c.startswith("PT__")
    or c.startswith("PM__")
    or c.startswith("TIPO__")
    or c.startswith("SUBTIPO__")
    or c.startswith("MOD__")
])

df_qa_final = pd.DataFrame(
    list(qa_final.items()),
    columns=["metrica", "valor"]
)

ruta_qa_final = carpeta_salida / "qa_super_merge_final.csv"

df_qa_final.to_csv(
    ruta_qa_final,
    index=False,
    encoding="utf-8-sig"
)

# También guardar una muestra para revisar rápido
ruta_muestra = carpeta_salida / "muestra_super_merge_1000_filas.csv"

df_sidpol_super_merge.head(1000).to_csv(
    ruta_muestra,
    index=False,
    encoding="utf-8-sig"
)

os.startfile(carpeta_salida)